# SDE-Net — errore per tipo di anomalia

MTGFlow assegna a ogni finestra uno score per entità (Eq. 14). La pipeline di
forecasting consuma solo lo score globale, quindi tutte le anomalie finiscono in
un unico gruppo `rare_or_extreme`. Qui la decomposizione viene recuperata per
distinguere **quale variabile fisica** ha reso la finestra improbabile.

La categoria primaria è il **driver dominante**, cioè l'entità con il contributo
maggiore a `S_c`:

| categoria | significato |
|---|---|
| `normal` | righe non anomale del 2019 |
| `solar` | contributo dominante di `solar_irradiance_poa` |
| `temperature` | contributo dominante di `temperature_2m` |
| `wind` | contributo dominante di `wind_speed_10m` |

Il numero di entità sopra la propria soglia Eq. 15 resta disponibile in
`n_entities_flagged` come **asse di severità separato** (`mode='severity'`).
Trattare le finestre multi-entità come categoria a sé (`mode='multi_split'`)
assorbiva circa un terzo delle anomalie e nascondeva il driver, quindi non è più
il default.

Nessun training viene rifatto: si usano `predictions.csv` della run SDE-Net e gli
score già prodotti dal detector.

**Limite dichiarato:** `S_ck` è una log-verosimiglianza negativa, quindi non ha
segno. `solar` comprende sia irraggiamento anomalmente basso (nuvole, polvere)
sia anomalmente alto. La separazione per segno richiede una climatologia
separata e non è fatta qui.

In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.anomaly_driver as anomaly_driver

anomaly_driver = importlib.reload(anomaly_driver)

RUN_NAME = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_seed1'
DETECTOR_SEED = 15
DETECTOR_ROOT = ROOT / 'outputs' / 'pvgis_mtgflow' / 'downstream_dense' / f'seed_{DETECTOR_SEED}'

out_dir = ROOT / 'outputs' / RUN_NAME
predictions_path = out_dir / 'predictions.csv'
entity_scores_path = DETECTOR_ROOT / 'entity_anomaly_scores.csv'
global_scores_path = DETECTOR_ROOT / 'anomaly_scores.csv'
labels_path = out_dir / anomaly_driver.DRIVER_LABELS_FILE

for required in (predictions_path, out_dir / 'reference_production_peaks.csv',
                 entity_scores_path, global_scores_path):
    if not required.is_file():
        raise FileNotFoundError(required)

print('Run SDE-Net :', out_dir)
print('Score entita:', entity_scores_path, f'({entity_scores_path.stat().st_size / 1e9:.1f} GB)')
print('Score globali:', global_scores_path)
print('Etichette   :', labels_path, '(gia presenti)' if labels_path.is_file() else '(da costruire)')

## 1. Etichette driver

Passo in streaming sul CSV per entità: viene tenuta solo una riga per finestra
anomala, quindi l'output è piccolo anche se l'input è di alcuni GB. Il passo è
lento (scan completo del file) ma va eseguito una sola volta: le celle
successive rileggono `entity_driver_labels.csv`.

In [ ]:
REBUILD_LABELS = False  # True forza il ricalcolo anche se il file esiste

if labels_path.is_file() and not REBUILD_LABELS:
    labels = anomaly_driver.load_driver_labels(labels_path)
    composition = (
        labels['category'].value_counts().rename_axis('category')
        .reset_index(name='n_windows')
    )
    composition['share'] = composition['n_windows'] / len(labels)
else:
    result = anomaly_driver.build_driver_labels(
        entity_scores_path,
        global_scores_path,
        out_path=labels_path,
        chunksize=2_000_000,
    )
    labels = result['labels']
    composition = result['composition']
    print('Entita rilevate:', result['entities'])

print(f'Finestre anomale etichettate: {len(labels):,}')
display(composition)

In [ ]:
score_columns = [c for c in labels.columns if c.startswith('score_')]
display(labels[score_columns].describe().T)
display(labels.head())

## 2. Controllo fisico sugli eventi noti

Verifica indipendente dell'attribuzione: gli eventi già analizzati nei notebook
dedicati devono risultare dominati dal driver atteso, e la quota va confrontata
con la baseline dell'intero 2019 (non con 1/3, perché i driver non sono
equifrequenti). L'evento polvere di aprile deve arricchire `solar`, l'ondata di
fine giugno `temperature`.

In [ ]:
event_days = pd.to_datetime(['2019-04-23', '2019-04-24', '2019-04-25', '2019-04-26',
                             '2019-06-28', '2019-06-29'])

baseline = labels['driver'].value_counts(normalize=True).rename('baseline_2019')
day = labels['timestamp'].dt.normalize()
selected = labels.loc[day.isin(event_days)].assign(day=day[day.isin(event_days)])
by_day = pd.crosstab(selected['day'], selected['driver'], normalize='index')

display(baseline.round(3).to_frame())
display(by_day.round(3))
# Arricchimento rispetto alla baseline: > 1 significa driver sovra-rappresentato.
display((by_day / baseline).round(2))

## 3. Confronto per driver

Stessa griglia dei notebook evento: tutte le righe diurne valide, nessun
campionamento, bin in percentuale della potenza di riferimento, statistiche
Tukey esatte. Cambia solo la categoria, che ora è il driver invece della data.

`unmatched_rare_rows` deve essere 0: se è positivo, `predictions.csv` e gli score
del detector non provengono dalla stessa run.

In [ ]:
CATEGORY_MODE = 'driver'  # 'driver' | 'multi_split' | 'severity'

driver_comparison = anomaly_driver.build_anomaly_driver_comparison_figures(
    out_dir,
    anomaly_driver.assign_categories(labels, mode=CATEGORY_MODE),
    figure_subdir=f'anomaly_driver_{CATEGORY_MODE}',
    metrics_name=f'anomaly_driver_{CATEGORY_MODE}_metrics.csv',
    chunksize=500_000,
)

metrics = driver_comparison['metrics']
print('Categorie   :', driver_comparison['categories'])
print('Righe rare non abbinate:', driver_comparison['unmatched_rare_rows'])
print('CSV metriche:', driver_comparison['metrics_path'])
display(metrics)

In [ ]:
for metric in ('mae', 'rmse', 'picp', 'nmpil', 'clc'):
    print(f'
=== {metric.upper()} ===')
    display(metrics.pivot(index='bin', columns='category', values=metric))

In [ ]:
normal = anomaly_driver.NORMAL_CATEGORY

# Degrado relativo rispetto allo strato normale, nello stesso bin.
wide = metrics.pivot(index='bin', columns='category', values='mae')
anomalous = [c for c in wide.columns if c != normal]
display(((wide[anomalous].div(wide[normal], axis=0) - 1.0) * 100.0).round(1))

# Coda pesante: RMSE/MAE alto = pochi errori molto grandi.
display(
    metrics.assign(tail=metrics['rmse'] / metrics['mae'])
    .pivot(index='bin', columns='category', values='tail').round(2)
)

# Quanto diluisce il gruppo unico rare_or_extreme: media pesata sui conteggi.
rare = metrics[metrics['category'] != normal].copy()
rare['w_mae'] = rare['mae'] * rare['count']
rare['w_picp'] = rare['picp'] * rare['count']
pooled = rare.groupby('bin')[['w_mae', 'w_picp', 'count']].sum()
pooled['mae_pooled'] = pooled['w_mae'] / pooled['count']
pooled['picp_pooled'] = pooled['w_picp'] / pooled['count']
pooled = pooled[['count', 'mae_pooled', 'picp_pooled']]
display(pooled.join(
    metrics[metrics['category'] == normal].set_index('bin')[['mae', 'picp']]
    .add_suffix('_normal')
).round(3))

In [ ]:
print('Grafici creati:', len(driver_comparison['figure_paths']))
for figure_path in driver_comparison['figure_paths'].values():
    display(Image(filename=str(figure_path)))

## 4. Come leggere il risultato

Domande a cui le tabelle rispondono:

- il degrado è concentrato su un driver o è uniforme? Se è concentrato, il gruppo
  unico `rare_or_extreme` sta diluendo l'effetto e la stratificazione per driver
  è la metrica corretta da riportare. La tabella `pooled` quantifica la
  diluizione;
- il rapporto RMSE/MAE distingue "errore diffusamente maggiore" da "pochi errori
  enormi": un rapporto molto sopra quello dello strato normale indica code
  pesanti, cioè transitori che il modello non vede arrivare;
- le anomalie di un driver si comportano come le normali? In tal caso il detector
  segnala condizioni senza impatto sulla previsione PV, e l'intervallo si allarga
  senza necessità;
- il PICP tiene su tutti i driver o cede solo su alcuni? È la lettura per driver
  della sotto-dispersione già osservata.

Con `CATEGORY_MODE='severity'` le stesse tabelle si leggono incrociando driver e
numero di entità coinvolte, per capire se le anomalie multi-entità sono più
difficili o solo più frequenti.

Attenzione alla numerosità: le categorie con poche centinaia di righe in un bin
danno metriche instabili. La colonna `count` va sempre riportata accanto al
valore.